# Pilot XAI-DSS trên IndPenSim — bản v3 (GPU)

So với v2:
- **bộ mô phỏng chạy trên GPU**, mô phỏng hàng trăm mẻ cùng lúc;
- **bộ điều khiển mức** áp dụng cho mọi chính sách, nên không mẻ nào bị tràn bình.

**Trước khi chạy:**
1. Upload `pilot_colab_v3.zip` và notebook này vào thư mục `MyDrive/De_Tai_Nong_San/`. Thư mục `pilot_v2/` của lần chạy trước phải còn ở đó, vì ô kiểm chứng cần kết quả CPU của v2.
2. Chọn *Runtime → Change runtime type → **T4 GPU***.
3. Chạy lần lượt từng ô. Lần đầu, JAX mất vài phút để biên dịch bộ mô phỏng; các bước sau chạy nhanh hơn nhiều.

## Ô 1. Kết nối Google Drive, giải nén code vào `pilot_v3/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/De_Tai_Nong_San'   # sửa nếu bạn để file ở thư mục khác
ZIP_PATH = os.path.join(DRIVE_DIR, 'pilot_colab_v3.zip')
WORK = os.path.join(DRIVE_DIR, 'pilot_v3')
V2_DIR = os.path.join(DRIVE_DIR, 'pilot_v2', 'runs', 'v2')   # kết quả CPU của v2, dùng để kiểm chứng

if not os.path.exists(os.path.join(WORK, 'jaxsim.py')):
    zipfile.ZipFile(ZIP_PATH).extractall(DRIVE_DIR)
    print('Đã giải nén vào', WORK)
os.chdir(WORK)
os.environ['PILOT_EXP'] = 'v3'      # kết quả ghi vào runs/v3/
print('Thư mục làm việc:', os.getcwd())
print('Có kết quả v2 để kiểm chứng:', os.path.exists(os.path.join(V2_DIR, 'results', 'batches.csv')))

## Ô 2. Kiểm tra GPU và thư viện

In [ ]:
!nvidia-smi -L
!pip install -q lightgbm shap
import jax, lightgbm, shap
print('JAX', jax.__version__, '| thiết bị:', jax.devices())
print('lightgbm', lightgbm.__version__, '| shap', shap.__version__)

## Ô 3. Kiểm chứng bộ mô phỏng GPU so với kết quả CPU của v2

So sản lượng của 24 mẻ chạy theo công thức, và từng dòng dữ liệu huấn luyện của 10 mẻ (đặc trưng và kết quả từng nhánh 24 giờ).

- Nếu dòng cuối in **"Bộ mô phỏng GPU khớp với kết quả CPU"** thì chạy tiếp.
- Nếu in **"CÓ SAI LỆCH"** thì dừng lại và gửi toàn bộ kết quả ô này cho Claude.

In [ ]:
!python -W ignore validate_gpu.py --v2-dir "{V2_DIR}"

## Ô 4. Chọn quy mô

In [ ]:
N_TRAIN, N_TEST = 120, 24      # giống v2 để so sánh; có thể tăng (ví dụ 240, 48) vì GPU chạy nhanh
print('Mẻ huấn luyện:', N_TRAIN, '| mẻ kiểm thử:', N_TEST)

## Ô 5. Sinh dữ liệu huấn luyện trên GPU (có bộ điều khiển mức)

In [ ]:
!python -W ignore gen_data_gpu.py --seeds {N_TRAIN}

## Ô 6. Huấn luyện mô hình dự báo và chọn hành động cố định tốt nhất (vài phút, chạy trên CPU)

In [ ]:
!python train_dss.py

## Ô 7. Chạy 4 chính sách trên các mẻ kiểm thử và kiểm chứng lời giải thích (GPU)

In [ ]:
!python -W ignore eval_dss_gpu.py --seeds {N_TEST}

## Ô 8. Tổng hợp số liệu

In [ ]:
!python analyze.py

## Ô 9. Gom kết quả thành `ket_qua_pilot_v3.zip` trên Drive

Tải file này về, đặt vào `D:\De_Tai_Nong_San\`, rồi báo Claude để phân tích.

In [ ]:
out = os.path.join(DRIVE_DIR, 'ket_qua_pilot_v3.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['runs/v3/results', 'runs/v3/models']:
        for root, _, files in os.walk(folder):
            for f in files:
                z.write(os.path.join(root, f))
    if os.path.exists('runs/v3/data/train.csv'):
        z.write('runs/v3/data/train.csv')
print('Đã lưu', out, f'({os.path.getsize(out)/1e6:.1f} MB)')